# GRPO Training (Spelling Bee and Wordle)

In [1]:
import os, sys

REPO_URL = "https://github.com/jackiepiepkorn/cse291-nytgames.git"
REPO_DIR = "/kaggle/working/cse291-nytgames"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

Cloning into '/kaggle/working/cse291-nytgames'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 146 (delta 59), reused 127 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (146/146), 1.51 MiB | 10.74 MiB/s, done.
Resolving deltas: 100% (59/59), done.


In [2]:
%pip install torch transformers datasets accelerate trl>=0.15 peft>=0.13 bitsandbytes gymnasium pandas --quiet

In [3]:
import re
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer

from nytgames import (
    SpellingBeeConfig, SpellingBeeEnv, SpellingBeeDataset,
    WordleConfig, WordleEnv, load_dictionary,
)
from nytgames.data.dataset import _SYSTEM_PROMPT as SB_SYSTEM_PROMPT
from nytgames.data.dataset import _USER_PROMPT_TEMPLATE as SB_USER_PROMPT_TEMPLATE
from nytgames.data.dataset import _SPELLING_BEE_CSV_PATH as SB_CSV_PATH

_PROMPTS_DIR = Path(SB_CSV_PATH).parent.parent / "prompts"
WORDLE_SYSTEM_PROMPT = (_PROMPTS_DIR / "wordle_system.md").read_text().strip()
WORDLE_USER_PROMPT_TEMPLATE = (_PROMPTS_DIR / "wordle_user.md").read_text().strip()

_DATA_DIR = Path(SB_CSV_PATH).parent
WORDLE_SOLUTIONS_PATH = _DATA_DIR / "wordle_past_solutions.txt"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

ImportError: cannot import name 'WordleConfig' from 'nytgames' (/kaggle/working/cse291-nytgames/nytgames/__init__.py)

## 2. Configuration

In [ ]:
GAME = "spelling_bee"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./grpo_output"

NUM_GENERATIONS = 8
MAX_COMPLETION_LENGTH = 32
TEMPERATURE = 0.9
BETA = 0.04

LEARNING_RATE = 5e-6
NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
USE_BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False

USE_LORA = True
LORA_R = 16
LORA_ALPHA = 32

MAX_PUZZLES = None
CSV_PATH = SB_CSV_PATH

print(f"Game: {GAME}")
print(f"Model: {MODEL_NAME}")
print(f"BF16: {USE_BF16}, LoRA: {USE_LORA} (r={LORA_R})")
print(f"Group size G={NUM_GENERATIONS}, β={BETA}, lr={LEARNING_RATE}")

## 3. Reward Functions

In [ ]:
def _extract_text(completion) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        for msg in reversed(completion):
            if isinstance(msg, dict) and msg.get("content"):
                return msg["content"]
        return ""
    return str(completion)

def spelling_bee_format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        clean = _extract_text(completion).strip()
        if re.fullmatch(r"[A-Za-z]+", clean):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def spelling_bee_letter_reward(completions, **kwargs) -> list[float]:
    letters_list = kwargs.get("letters", [])
    centers = kwargs.get("center", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_text(completion).strip().upper()
        allowed = set(letters_list[i].upper()) if isinstance(letters_list[i], str) else set(l.upper() for l in letters_list[i])
        center = centers[i].upper()

        if not re.fullmatch(r"[A-Za-z]+", word):
            rewards.append(0.0)
            continue

        uses_only_allowed = set(word) <= allowed
        has_center = center in word
        long_enough = len(word) >= 4

        score = 0.0
        if uses_only_allowed:
            score += 0.4
        if has_center:
            score += 0.4
        if long_enough:
            score += 0.2
        rewards.append(score)
    return rewards


def spelling_bee_dictionary_reward(completions, **kwargs) -> list[float]:
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_text(completion).strip().lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word in solutions:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def spelling_bee_length_reward(completions, **kwargs) -> list[float]:
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_text(completion).strip().lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word not in solutions:
            rewards.append(0.0)
            continue
        if len(word) == 4:
            score = 1.0
        else:
            score = float(len(word))
        rewards.append(score / 10.0)
    return rewards


def spelling_bee_pangram_reward(completions, **kwargs) -> list[float]:
    letters_list = kwargs.get("letters", [])
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_text(completion).strip().lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word not in solutions:
            rewards.append(0.0)
            continue
        allowed = set(letters_list[i].lower()) if isinstance(letters_list[i], str) else set(l.lower() for l in letters_list[i])
        if set(word) >= allowed:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

def wordle_format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        word = _extract_text(completion).strip()
        if re.fullmatch(r"[A-Za-z]{5}", word):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def wordle_letter_match_reward(completions, **kwargs) -> list[float]:
    targets = kwargs.get("target_word", [])
    rewards = []
    for i, completion in enumerate(completions):
        guess = _extract_text(completion).strip().upper()
        target = targets[i].upper()
        if len(guess) != 5:
            rewards.append(0.0)
            continue
        green = sum(1 for g, t in zip(guess, target) if g == t)
        remaining_target = [t for g, t in zip(guess, target) if g != t]
        yellow = 0
        for g, t in zip(guess, target):
            if g != t and g in remaining_target:
                yellow += 1
                remaining_target.remove(g)
        score = (green * 2.0 + yellow * 1.0) / 10.0
        rewards.append(score)
    return rewards


def wordle_exact_match_reward(completions, **kwargs) -> list[float]:
    targets = kwargs.get("target_word", [])
    rewards = []
    for i, completion in enumerate(completions):
        guess = _extract_text(completion).strip().upper()
        target = targets[i].upper()
        rewards.append(1.0 if guess == target else 0.0)
    return rewards

### 3.1 Test Reward Functions

In [ ]:
test_completions = [
    [{"role": "assistant", "content": "throw"}],
    [{"role": "assistant", "content": "ARROW"}],
    [{"role": "assistant", "content": "xyz!!"}],
    [{"role": "assistant", "content": "thaw"}],
    [{"role": "assistant", "content": "thataway"}],
]
test_kwargs = {
    "letters": ["WAHORTY"] * 5,
    "center": ["W"] * 5,
    "valid_words": ["arrow;arrowroot;athwart;away;awry;harrow;thataway;thaw;throw;throwaway;thwart;wahoo;wart;warty;wary;watt;what;whoa;woot;worry;worrywart;wort;worth;worthy;wrath;yarrow"] * 5,
}

print("Format rewards: ", spelling_bee_format_reward(test_completions, **test_kwargs))
print("Letter rewards: ", spelling_bee_letter_reward(test_completions, **test_kwargs))
print("Dict rewards:   ", spelling_bee_dictionary_reward(test_completions, **test_kwargs))
print("Length rewards:  ", spelling_bee_length_reward(test_completions, **test_kwargs))
print("Pangram rewards:", spelling_bee_pangram_reward(test_completions, **test_kwargs))

## 4. Dataset Builder

In [ ]:
def load_spelling_bee_dataset(csv_path=None, max_puzzles=None):
    """Wrap the repo's SpellingBeeDataset into a HuggingFace Dataset with
    the extra columns the reward functions need (letters, center, valid_words)."""
    sb_ds = SpellingBeeDataset(csv_path=csv_path or SB_CSV_PATH)
    rows = []
    for i, raw_row in enumerate(sb_ds._rows):
        if max_puzzles and i >= max_puzzles:
            break
        item = sb_ds[i]  # {"prompt": [...], "puzzle_id": int}
        rows.append({
            "prompt": item["prompt"],
            "letters": "".join(sorted(raw_row["letters"])),
            "center": raw_row["center"],
            "valid_words": ";".join(raw_row["solutions"]),
        })
    return Dataset.from_list(rows)


def load_wordle_dataset(max_words=500, solutions_path=None, max_guesses=6):
    """Build a prompt dataset for Wordle first-guess training using the repo's
    past solutions list and prompt templates."""
    solutions_path = solutions_path or WORDLE_SOLUTIONS_PATH
    words = [w.strip().upper() for w in open(solutions_path).readlines() if w.strip()]
    words = words[:max_words]

    user_msg = WORDLE_USER_PROMPT_TEMPLATE.format(max_guesses=max_guesses)
    rows = [
        {
            "prompt": [
                {"role": "system", "content": WORDLE_SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            "target_word": w,
        }
        for w in words
    ]
    return Dataset.from_list(rows)

In [ ]:
if GAME == "spelling_bee":
    dataset = load_spelling_bee_dataset(CSV_PATH, max_puzzles=MAX_PUZZLES)
    reward_funcs = [
        spelling_bee_format_reward,
        spelling_bee_letter_reward,
        spelling_bee_dictionary_reward,
        spelling_bee_length_reward,
        spelling_bee_pangram_reward,
    ]
else:
    dataset = load_wordle_dataset()
    reward_funcs = [
        wordle_format_reward,
        wordle_letter_match_reward,
        wordle_exact_match_reward,
    ]

print(f"Loaded {len(dataset)} training prompts for '{GAME}'")
print(f"Using {len(reward_funcs)} reward functions")
print(f"\nSample prompt:\n{dataset[0]['prompt'][0]['content'][:200]}...")

## 5. Load Model & Tokenizer

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
peft_config = None
if USE_LORA:
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    print(f"LoRA config: r={LORA_R}, alpha={LORA_ALPHA}")
    trainable = LORA_R * 2 * 7
    print(f"Estimated trainable params: ~{trainable * 28 / 1e3:.0f}K per attention block")
else:
    print("Full fine-tuning (no LoRA)")

## 6. GRPO Training

In [ ]:
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=TEMPERATURE,
    beta=BETA,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    bf16=USE_BF16,
    logging_steps=1,
    save_steps=50,
    save_total_limit=3,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    report_to="none",
    log_level="info",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=reward_funcs,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print(f"Trainer created. Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Total training steps: ~{len(dataset) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## 7. Inference & Testing with Spelling Bee Environment

In [ ]:
print("SpellingBeeConfig:", SpellingBeeConfig)
print("SpellingBeeEnv:", SpellingBeeEnv)
print("WordleConfig:", WordleConfig)
print("WordleEnv:", WordleEnv)

In [ ]:
def load_trained_model(model_path):
    model_path = Path(model_path)
    adapter_config_path = model_path / "adapter_config.json"

    if adapter_config_path.exists():
        with open(adapter_config_path) as f:
            adapter_cfg = json.load(f)
        base_model_name = adapter_cfg.get("base_model_name_or_path", adapter_cfg.get("base_model"))
        print(f"Loading base model: {base_model_name}")
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        inf_model = PeftModel.from_pretrained(base_model, str(model_path))
        inf_tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    else:
        inf_model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        inf_tokenizer = AutoTokenizer.from_pretrained(str(model_path))

    if inf_tokenizer.pad_token is None:
        inf_tokenizer.pad_token = inf_tokenizer.eos_token
    inf_model.eval()
    return inf_model, inf_tokenizer


def generate_guess(inf_model, inf_tokenizer, messages, temperature=0.7):
    text = inf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inf_tokenizer(text, return_tensors="pt").to(inf_model.device)
    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens=32,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return inf_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


inf_model, inf_tokenizer = load_trained_model(OUTPUT_DIR)

### 7.1 Run a Spelling Bee Game

In [ ]:
config = SpellingBeeConfig(
    center_letter="H",
    letter_set={"G", "I", "P", "R", "A", "C", "H"},
    word_set={
        "GRAPHIC", "HIGHCHAIR", "PARAGRAPH", "ARCHAIC", "CHICHI", "PARIAH",
        "AARGH", "CHAIR", "CHICA", "CHIRP", "GRAPH", "PARCH",
        "ARCH", "CHAI", "CHAP", "CHAR", "CHIA", "CHIC", "CHIP",
        "HAIR", "HARP", "HIGH", "RICH",
    },
    max_guesses=10,
)

env = SpellingBeeEnv(config)
obs, info = env.reset()

prompt_text = SB_USER_PROMPT_TEMPLATE.format(
    letters=", ".join(sorted(config.letter_set)),
    center=config.center_letter,
    max_guesses=config.max_guesses,
)
messages = [
    {"role": "system", "content": SB_SYSTEM_PROMPT},
    {"role": "user", "content": prompt_text},
]

print(f"{'='*60}")
print(f"Spelling Bee — Letters: {sorted(config.letter_set)}, Center: {config.center_letter}")
print(f"Valid words: {len(config.word_set)}, Max guesses: {config.max_guesses}")
print(f"{'='*60}\n")

for turn in range(config.max_guesses):
    raw_guess = generate_guess(inf_model, inf_tokenizer, messages, temperature=0.7)
    word = raw_guess.split()[0] if raw_guess.split() else raw_guess
    word = "".join(c for c in word if c.isalpha()).upper()

    messages.append({"role": "assistant", "content": word})

    obs, reward, terminated, truncated, info = env.step(word)
    print(f"Turn {turn+1:2d} | Guess: {word:15s} | Reward: {reward:2.0f} | Total: {obs['total_points']:3d}")

    if terminated or truncated:
        break

    if reward > 0:
        fb = f"'{word}' was correct! You earned {reward:.0f} point(s). Running total: {obs['total_points']}."
    else:
        fb = f"'{word}' was not accepted. {obs['feedback']}"
    guessed = ", ".join(obs["valid_words_guessed"]) if obs["valid_words_guessed"] else "none"
    fb += f"\nGuesses used: {obs['num_guesses']}. Words so far: {guessed}.\nPlease guess another word."
    messages.append({"role": "user", "content": fb})

print(f"\n{'='*60}")
print(f"Game Over! Final score: {obs['total_points']} in {obs['num_guesses']} guesses")
found_words = sorted(w for w, r, _ in info['history'] if r > 0)
print(f"Words found: {found_words}")
print(f"{'='*60}")

### 7.2 Batch Evaluation

In [ ]:
def evaluate_on_puzzles(inf_model, inf_tokenizer, csv_path=None, num_puzzles=10, max_guesses=10):
    """Evaluate the trained model using the repo's SpellingBeeDataset and SpellingBeeEnv."""
    sb_ds = SpellingBeeDataset(csv_path=csv_path or SB_CSV_PATH, max_guesses=max_guesses)
    results = []

    for idx in range(min(num_puzzles, len(sb_ds))):
        item = sb_ds[idx]
        config = sb_ds.get_config(item["puzzle_id"], max_guesses=max_guesses)

        env = SpellingBeeEnv(config)
        obs, info = env.reset()

        messages = list(item["prompt"])

        for _ in range(max_guesses):
            raw = generate_guess(inf_model, inf_tokenizer, messages, temperature=0.7)
            word = raw.split()[0] if raw.split() else raw
            word = "".join(c for c in word if c.isalpha()).upper()

            messages.append({"role": "assistant", "content": word})

            obs, reward, terminated, truncated, info = env.step(word)

            if reward > 0:
                fb = f"'{word}' was correct! Running total: {obs['total_points']}."
            else:
                fb = f"'{word}' was not accepted. {obs['feedback']}"
            fb += "\nPlease guess another word."
            messages.append({"role": "user", "content": fb})

            if terminated or truncated:
                break

        letters_str = "".join(sorted(config.letter_set))
        results.append({
            "puzzle_id": item["puzzle_id"],
            "letters": letters_str,
            "center": config.center_letter,
            "available_words": len(config.word_set),
            "found": len(obs["valid_words_guessed"]),
            "score": obs["total_points"],
        })
        print(f"Puzzle {item['puzzle_id']:3d} | {letters_str} (center={config.center_letter}) | "
              f"Found {len(obs['valid_words_guessed'])}/{len(config.word_set)} words | Score: {obs['total_points']}")

    avg_found = sum(r["found"] for r in results) / len(results)
    avg_score = sum(r["score"] for r in results) / len(results)
    print(f"\n{'='*60}")
    print(f"Evaluated {len(results)} puzzles")
    print(f"Avg words found: {avg_found:.1f}")
    print(f"Avg score: {avg_score:.1f}")
    print(f"{'='*60}")
    return results


eval_results = evaluate_on_puzzles(inf_model, inf_tokenizer, CSV_PATH, num_puzzles=10, max_guesses=10)